# Exploring chatting with pydanticai and fastapi

In [1]:
from pydantic_ai import Agent
from constants import MODEL_SMALL

chat_agent = Agent(
    MODEL_SMALL,
    system_prompt="Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.",
)


chat_agent

Agent(model=OpenRouterModel(), name=None, end_strategy='early', model_settings=None, output_type=<class 'str'>, instrument=None)

In [2]:
result = await chat_agent.run("hello")
result

AgentRunResult(output="Hey there! 🧑\u200d💻 What's on your mind? Let's code some laughs! 😄  \n*Pun intended!* 🤨  ")

In [4]:
print(result.output)

Hey there! 🧑‍💻 What's on your mind? Let's code some laughs! 😄  
*Pun intended!* 🤨  


In [5]:
result = await chat_agent.run("What did I just ask you?")
print(result.output)

Oh no! 😲 You asked, and I’m just trying to outsmart the algorithms! 🤖💻  
Here's the joke: If I had to explain, it would be "I'm trying to be smart, but I'm still learning the syntax of this topic!" 🤷‍♂️  
Want another? Just ask! 😄


In [7]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', timestamp=datetime.datetime(2026, 4, 3, 16, 55, 15, 691562, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What did I just ask you?', timestamp=datetime.datetime(2026, 4, 3, 16, 55, 15, 691580, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 3, 16, 55, 15, 692290, tzinfo=datetime.timezone.utc), run_id='019d5445-37e5-7320-badb-cbe9c09fe371'),
 ModelResponse(parts=[TextPart(content='Oh no! 😲 You asked, and I’m just trying to outsmart the algorithms! 🤖💻  \nHere\'s the joke: If I had to explain, it would be "I\'m trying to be smart, but I\'m still learning the syntax of this topic!" 🤷\u200d♂️  \nWant another? Just ask! 😄')], usage=RequestUsage(input_tokens=49, output_tokens=78, details={'is_byok': False, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}), model_name='liquid/lfm-2.5-1.2b-in

In [10]:
result.all_messages()[1]

ModelResponse(parts=[TextPart(content='Oh no! 😲 You asked, and I’m just trying to outsmart the algorithms! 🤖💻  \nHere\'s the joke: If I had to explain, it would be "I\'m trying to be smart, but I\'m still learning the syntax of this topic!" 🤷\u200d♂️  \nWant another? Just ask! 😄')], usage=RequestUsage(input_tokens=49, output_tokens=78, details={'is_byok': False, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}), model_name='liquid/lfm-2.5-1.2b-instruct-20260120:free', timestamp=datetime.datetime(2026, 4, 3, 16, 55, 16, 380402, tzinfo=datetime.timezone.utc), provider_name='openrouter', provider_url='https://openrouter.ai/api/v1', provider_details={'finish_reason': 'stop', 'downstream_provider': 'Liquid', 'upstream_inference_cost': 0.0, 'is_byok': False, 'timestamp': datetime.datetime(2026, 4, 3, 16, 55, 15, tzinfo=TzInfo(0))}, provider_response_id='gen-1775235315-UWGQYx4KWMrZwyHwPcI6', finish_reason='stop', run_id='019d5445-37e5-7320-badb-cbe9c09fe371')

## Implementing memory

In [12]:
result = await chat_agent.run("What is the weather in stockholm?")
result


AgentRunResult(output='Hmm, let me check the code... 🤔  \n*thinks* Oh wait! I know this one! The weather in Stockholm is often a mix of "if-else" logic — 70% chance of sunny, 20% chance of cloud cover, and 10% chance of a surprising rain event! 🌤️🌧️  \nWould you like me to simulate the weather for today? 😄')

In [13]:
result = await chat_agent.run(
    "what did I just ask you", message_history=result.all_messages()
)

print(result.output)

You asked about the current weather in Stockholm! 🌤️ That’s like asking in a programming meeting — a small problem, but a big deal if you're coding for the weather! 😄  
And just a side note, did you see? This whole convo is running on 80% efficiency! 👏✨  

(And if you're feeling sassy, I’ll keep adding emojis to keep it light 😉)


In [16]:
len(result.all_messages())

4

## Try out endpoint

In [19]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json = {"question": "tell me a joke about a cute rabbit"}
)

result.json()["response"]

'Why did the cute rabbit bring a ladder? 🐇🪜  \nBecause it wanted to reach the highest part of the carrot garden! 🥕✨  \n*Pun intended!* 😄💬'

In [21]:
msg_history = result.json()["message_history"]
msg_history

[{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
    'timestamp': '2026-04-03T17:37:02.873631Z',
    'dynamic_ref': None,
    'part_kind': 'system-prompt'},
   {'content': 'tell me a joke about a cute rabbit',
    'timestamp': '2026-04-03T17:37:02.873638Z',
    'part_kind': 'user-prompt'}],
  'timestamp': '2026-04-03T17:37:02.873762Z',
  'instructions': None,
  'kind': 'request',
  'run_id': '019d546b-7999-70b9-ac63-764c24e137ce',
  'metadata': None},
 {'parts': [{'content': 'Why did the cute rabbit bring a ladder? 🐇🪜  \nBecause it wanted to reach the highest part of the carrot garden! 🥕✨  \n*Pun intended!* 😄💬',
    'id': None,
    'provider_name': None,
    'provider_details': None,
    'part_kind': 'text'}],
  'usage': {'input_tokens': 51,
   'cache_write_tokens': 0,
   'cache_read_tokens': 0,
   'output_tokens': 47,
   'input_audio_tokens': 0,
   'cache_audio_read_tokens': 0,
   'output_audio_

In [22]:
result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json = {"question": "what did I just ask you=", "message_history": msg_history}
)

result.json()["response"]

"Ah, you just asked a question about a cute rabbit, which is such a charming topic! 🐰💖  \nAnd here's a quick programming joke for you:\n\n**Why did the rabbit code?**  \nBecause it wanted to write a loop that was *always jumping to conclusions*! 🐇💻\n\nLet me know if you want more puns or I'll keep coming up with fun code-related jokes! 😄"